In [1]:
print('hello')

hello


In [ ]:
print('lib importing...')
import json
import os

import joblib
import optuna
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

print('importing completed')


lib importing...
importing completed


In [ ]:
# =========================================================
# 0. 경로 설정
# =========================================================
data_dir = 'C:/potenup3/pj02_Daangn-marke/data/processed'
model_dir = 'C:/potenup3/pj02_Daangn-marke/model'
os.makedirs(model_dir, exist_ok=True)

train_path = os.path.join(data_dir, 'train_feature_v21.csv')
valid_path = os.path.join(data_dir, 'valid_feature_v21.csv')
test_path = os.path.join(data_dir, 'test_feature_v21.csv')

model_path = os.path.join(model_dir, 'lgbm_sold_predictor_v21.pkl')
feature_importance_path = os.path.join(model_dir, 'lgbm_feature_importance_v21.csv')
metrics_path = os.path.join(model_dir, 'lgbm_metrics_v21.json')
best_params_path = os.path.join(model_dir, 'lgbm_best_params_v21.json')


# =========================================================
# 1. 데이터 로드
# =========================================================
print('[INFO] Loading train/valid/test...')
train_df = pd.read_csv(train_path, low_memory=False)
valid_df = pd.read_csv(valid_path, low_memory=False)
test_df = pd.read_csv(test_path, low_memory=False)

print('[INFO] train shape:', train_df.shape)
print('[INFO] valid shape:', valid_df.shape)
print('[INFO] test shape :', test_df.shape)


# =========================================================
# 2. feature / target 분리
# =========================================================
target_col = 'sold'
drop_cols = ['id', target_col]

X_train = train_df.drop(columns=drop_cols).copy()
y_train = train_df[target_col].copy()

X_valid = valid_df.drop(columns=drop_cols).copy()
y_valid = valid_df[target_col].copy()

X_test = test_df.drop(columns=drop_cols).copy()
y_test = test_df[target_col].copy()

print('[INFO] Number of features:', X_train.shape[1])


# =========================================================
# 3. categorical feature 지정
# =========================================================
categorical_cols = [
    'feat_price_bucket',
    'feat_brandName',
    'feat_seller_temp_bucket',
    'feat_label',
    'feat_coarse_label',
    'feat_top1_label',
    'feat_region_id',
]

# categorical 컬럼: 먼저 문자열 + unknown 처리 후 category 변환
for col in categorical_cols:
    if col in X_train.columns:
        X_train[col] = (
            X_train[col]
            .astype(str)
            .fillna('unknown')
            .replace('nan', 'unknown')
            .astype('category')
        )
        X_valid[col] = (
            X_valid[col]
            .astype(str)
            .fillna('unknown')
            .replace('nan', 'unknown')
            .astype('category')
        )
        X_test[col] = (
            X_test[col]
            .astype(str)
            .fillna('unknown')
            .replace('nan', 'unknown')
            .astype('category')
        )

# 나머지 컬럼은 숫자형 처리
for col in X_train.columns:
    if col not in categorical_cols:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_valid[col] = pd.to_numeric(X_valid[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# 숫자형 결측은 train median 기준으로 보정
for col in X_train.columns:
    if col not in categorical_cols:
        median_value = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_value)
        X_valid[col] = X_valid[col].fillna(median_value)
        X_test[col] = X_test[col].fillna(median_value)


# =========================================================
# 4. 평가 함수
# =========================================================
def evaluate_binary(y_true, y_pred, y_prob):
    return {
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred)),
        'recall': float(recall_score(y_true, y_pred)),
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
    }


# =========================================================
# 5. Optuna objective
# =========================================================
def objective(trial):
    params = {
        'objective': 'binary',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'class_weight': 'balanced',
        'n_jobs': -1,
        # 탐색 대상
        'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    model = LGBMClassifier(**params)
    # LightGBM의 early_stopping과 log_evaluation 콜백을 활용하여 검증 AUC를 모니터링하면서 학습
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='auc',
        categorical_feature=categorical_cols,
        callbacks=[
            early_stopping(stopping_rounds=100, verbose=False),
            log_evaluation(0),
        ],
    )

    valid_pred_proba = model.predict_proba(X_valid)[:, 1]
    valid_auc = roc_auc_score(y_valid, valid_pred_proba)

    return valid_auc


# =========================================================
# 6. Optuna 탐색
# =========================================================
print('[INFO] Start Optuna tuning...')

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print('[INFO] Best trial:', study.best_trial.number)
print('[INFO] Best AUC:', study.best_value)
print('[INFO] Best params:', study.best_params)

# best params 저장
with open(best_params_path, 'w', encoding='utf-8') as f:
    json.dump(study.best_params, f, ensure_ascii=False, indent=2)

print(f'[INFO] Best params saved -> {best_params_path}')


# =========================================================
# 7. 최적 파라미터로 최종 모델 학습
# =========================================================
best_params = study.best_params.copy()
best_params.update(
    {
        'objective': 'binary',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'class_weight': 'balanced',
        'n_jobs': -1,
    }
)

print('[INFO] Training final model with best params...')
model = LGBMClassifier(**best_params)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric='auc',
    categorical_feature=categorical_cols,
    callbacks=[
        early_stopping(stopping_rounds=100, verbose=False),
        log_evaluation(100),
    ],
)

print('[INFO] Final training done.')


# =========================================================
# 8. 예측
# =========================================================
valid_pred_proba = model.predict_proba(X_valid)[:, 1]
test_pred_proba = model.predict_proba(X_test)[:, 1]

# 기본 threshold = 0.5
valid_pred = (valid_pred_proba >= 0.5).astype(int)
test_pred = (test_pred_proba >= 0.5).astype(int)


# =========================================================
# 9. 평가
# =========================================================
valid_metrics = evaluate_binary(y_valid, valid_pred, valid_pred_proba)
test_metrics = evaluate_binary(y_test, test_pred, test_pred_proba)

print('\n[VALID METRICS]')
for k, v in valid_metrics.items():
    print(k, ':', v)

print('\n[TEST METRICS]')
for k, v in test_metrics.items():
    print(k, ':', v)

print('\n[TEST CLASSIFICATION REPORT]')
print(classification_report(y_test, test_pred, digits=4))


# =========================================================
# 10. Feature Importance 저장
# =========================================================
feature_importance_df = pd.DataFrame(
    {'feature': X_train.columns, 'importance': model.feature_importances_}
).sort_values('importance', ascending=False)

feature_importance_df.to_csv(feature_importance_path, index=False, encoding='utf-8-sig')
print(f'[INFO] Feature importance saved -> {feature_importance_path}')

print('\n[TOP 20 FEATURE IMPORTANCE]')
print(feature_importance_df.head(20))


# =========================================================
# 11. 모델 저장
# =========================================================
joblib.dump(model, model_path)
print(f'[INFO] Model saved -> {model_path}')


# =========================================================
# 12. 평가 지표 저장
# =========================================================
all_metrics = {
    'valid': valid_metrics,
    'test': test_metrics,
    'n_features': int(X_train.shape[1]),
    'categorical_features': categorical_cols,
    'best_params': study.best_params,
    'best_valid_auc': study.best_value,
}

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=2)

print(f'[INFO] Metrics saved -> {metrics_path}')
print('[INFO] All done.')

[INFO] Loading train/valid/test...
[INFO] train shape: (170044, 36)
[INFO] valid shape: (21256, 36)
[INFO] test shape : (21256, 36)
[INFO] Number of features: 34


[I 2026-03-08 23:25:20,280] A new study created in memory with name: no-name-570c9dad-d33c-4d84-9f4c-508cd5807353


[INFO] Start Optuna tuning...
[LightGBM] [Info] Number of positive: 54022, number of negative: 116022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007233 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3165
[LightGBM] [Info] Number of data points in the train set: 170044, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[I 2026-03-08 23:25:56,254] Trial 0 finished with value: 0.7934426246916199 and parameters: {'n_estimators': 1755, 'learning_rate': 0.014785737529488524, 'num_leaves': 93, 'min_child_samples': 31, 'subsample': 0.6211685914279271, 'colsample_bytree': 0.7621122210733663, 'reg_alpha': 0.4239252156712929, 'reg_lambda': 0.005923158952721477}. Best is trial 0 with value: 0.7934426246916199.


[LightGBM] [Info] Number of positive: 54022, number of negative: 116022
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009321 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3165
[LightGBM] [Info] Number of data points in the train set: 170044, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


...

[LightGBM] [Info] Number of positive: 54022, number of negative: 116022
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008940 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3165
[LightGBM] [Info] Number of data points in the train set: 170044, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[I 2026-03-08 23:39:18,165] Trial 28 finished with value: 0.8025422396867414 and parameters: {'n_estimators': 1201, 'learning_rate': 0.07704699575750328, 'num_leaves': 119, 'min_child_samples': 55, 'subsample': 0.8560313439189057, 'colsample_bytree': 0.7106405751978091, 'reg_alpha': 2.5943742322241836, 'reg_lambda': 9.580964539318812e-07}. Best is trial 18 with value: 0.8054761445364036.


[LightGBM] [Info] Number of positive: 54022, number of negative: 116022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006889 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3165
[LightGBM] [Info] Number of data points in the train set: 170044, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[I 2026-03-08 23:39:50,968] Trial 29 finished with value: 0.7851871596616821 and parameters: {'n_estimators': 1998, 'learning_rate': 0.014734101850250649, 'num_leaves': 35, 'min_child_samples': 85, 'subsample': 0.7940028660265177, 'colsample_bytree': 0.7611525712103467, 'reg_alpha': 0.22240737066444052, 'reg_lambda': 0.0023328093767901984}. Best is trial 18 with value: 0.8054761445364036.


[INFO] Best trial: 18
[INFO] Best AUC: 0.8054761445364036
[INFO] Best params: {'n_estimators': 1945, 'learning_rate': 0.04611819108893845, 'num_leaves': 110, 'min_child_samples': 100, 'subsample': 0.7179171599911441, 'colsample_bytree': 0.803520582003886, 'reg_alpha': 0.8577660012305288, 'reg_lambda': 0.00011981000704943269}
[INFO] Best params saved -> C:/potenup3/pj02_Daangn-marke/model\lgbm_best_params_v21.json
[INFO] Training final model with best params...
[LightGBM] [Info] Number of positive: 54022, number of negative: 116022
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011544 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3165
[LightGBM] [Info] Number of data points in the train set: 170044, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[100]	valid_0's auc: 0.775977	valid_0's